# 【講師用】Ch.4 機械学習モデル構築（scikit-learn）完全解説

> 🔒 **このノートブックは講師専用です。受講者には配布しないでください。**

---

## 📢 講師向け冒頭ガイド

### このチャプターで最も伝えたいこと
1. **データリーク防止**：`scaler.fit` は訓練データのみ。これは絶対に外せない
2. **正解率だけで判断しない**：混同行列で「どう間違えたか」を確認する
3. **EDAの仮説を特徴量重要度で検証する**：Ch.2 で「proline が重要そう」→ 答え合わせ

### よくある受講者の躓きポイント
| 躓き | 対処法 |
|------|--------|
| `fit_transform` と `transform` の違い | 「試験の採点基準を学習データだけで決める。テストに採点基準を教えてはいけない」と例える |
| 混同行列の行と列の向きがわからない | 「行（縦）= 正解ラベル、列（横）= 予測ラベル」と図で示す |
| Precision と Recall の違いが覚えられない | 「Precision = 予測した中の正解率、Recall = 正解の中の検出率」と例で説明 |

### 時間配分目安（座学20分）
- 4.1 データ分割：4分
- 4.2 前処理・スケーリング：5分
- 4.3 モデル学習：3分
- 4.4 評価（正解率・混同行列）：5分
- 4.5 特徴量重要度：3分

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# データ準備
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

X = df[wine.feature_names]  # 特徴量（13列）
y = df['target']             # ターゲット（品種: 0/1/2）

print('特徴量の形状:', X.shape)
print('ターゲットの分布:')
print(y.value_counts().sort_index())

---
## 4.1 データ分割（Train / Test）

### 📢 講師メモ
「なぜ分割するか」を必ず先に説明してからコードを見せてください。
具体的な例えが効果的です：
> 「受験勉強で練習問題を何度も解いて、試験当日は全く同じ問題が出た。
> 正解できたとして、それは本当に実力があるといえますか？」

In [ ]:
# ─── データ分割の仕組みを図で説明 ────────────────────────────────
#
# 全データ（178件）
# ┌─────────────────────────────────┬──────────┐
# │     訓練データ（80% = 142件）      │テストデータ │
# │     モデルの学習に使う             │（20% = 36件）│
# │                                  │評価専用    │
# └─────────────────────────────────┴──────────┘

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% をテストデータに
    random_state=42,  # 乱数シード固定（再現性確保）
    stratify=y        # 品種の比率を訓練・テストで均等に保つ
)

print('訓練データ:', X_train.shape)
print('テストデータ:', X_test.shape)
print()
print('訓練データの品種分布:')
print(y_train.value_counts().sort_index())
print()
print('テストデータの品種分布:')
print(y_test.value_counts().sort_index())

### 📢 `stratify=y` の説明
```
stratify を使わないと：
  たまたまテストデータに品種 0 が多く入ってしまうことがある
  → 偏った評価になる

stratify=y を使うと：
  訓練データとテストデータで品種の比率が均等に保たれる
  → 公平な評価ができる
```

> 「必ず `stratify=y` を使う習慣をつけてください。これを忘れると評価が不正確になります。」

---
## 4.2 前処理：標準化（StandardScaler）

### 📢 講師メモ
**データリーク（Data Leakage）** の説明が最重要です。

> 「テストデータでもfit（学習）することを『データリーク』といいます。
> テストデータの情報がモデルに漏れてしまう（リーク）ことで、評価が甘くなります。
> 本番環境では存在しないデータで評価した数字を信頼してしまう危険があります。」

In [ ]:
# スケールの差を確認（なぜスケーリングが必要か）
print('スケーリング前の値の範囲：')
print(X_train[['alcohol', 'proline']].describe().loc[['min', 'max']].round(1))

### 📢 スケーリングが必要な理由
```
alcohol の範囲: 11.0 〜 14.8  (差: 約3.8)
proline の範囲: 278 〜 1680   (差: 約1400)

距離ベースのアルゴリズム（KNN・SVM等）では
proline の差が alcohol の差の400倍以上なので、
proline が不当に支配的な影響を持つ。

ランダムフォレストは厳密にはスケーリング不要だが、
「標準的なワークフローとして習慣化する」目的で行う。
```

In [ ]:
# ─── StandardScaler の使い方（最重要ルール付き）─────────────────

scaler = StandardScaler()

# ✅ 正しい使い方
# fit_transform：平均・標準偏差を「訓練データから」学習して変換
X_train_scaled = scaler.fit_transform(X_train)

# transform のみ：テストデータには「訓練データの平均・標準偏差」を使って変換
X_test_scaled  = scaler.transform(X_test)

# スケーリング後の確認
print('スケーリング後の値（訓練データ）：')
X_train_df = pd.DataFrame(X_train_scaled, columns=wine.feature_names)
print(X_train_df[['alcohol', 'proline']].describe().loc[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
# ─── NG パターンの実演（やってはいけない例）───────────────────────
# ⚠️ 以下はデータリークの例。実際には絶対にやらない！

scaler_ng = StandardScaler()
X_test_leaked = scaler_ng.fit_transform(X_test)  # ← テストデータでfit はNG！

# この場合、テストデータの「平均・標準偏差」がモデルに使われる
# → 本番環境では存在しない情報が混入し、評価が過度に楽観的になる

print('⚠️ これはデータリークです。絶対に使用しないこと。')
print('テストデータでfit した mean:', scaler_ng.mean_[:2].round(3))
print('訓練データでfit した mean:', scaler.mean_[:2].round(3))
print('値が異なります（=異なる基準でスケーリングされている）')

---
## 4.3 モデル学習（ランダムフォレスト）

### 📢 講師メモ
ランダムフォレストの仕組みは「多数決」で十分です。
「なぜ決定木を1本じゃなくてたくさん使うか」をアニメーションのイメージで説明しましょう。

In [ ]:
# ランダムフォレストの概念図
print('=== ランダムフォレストの仕組み ===')
print()
print('　　サンプルA → 決定木1 → Barolo')
print('　　サンプルA → 決定木2 → Barolo')
print('　　サンプルA → 決定木3 → Grignolino')
print('　　　　　　　　　　　　　↓')
print('　　　　　多数決: Barolo（2対1）→ 最終予測: Barolo')
print()
print('ポイント：')
print('  各決定木は「異なるサンプル」「異なる特徴量」で学習する')
print('  → バラバラな判断をする木が多数決することで、全体の精度が上がる')

# モデルの定義と学習
model = RandomForestClassifier(
    n_estimators=100,  # 決定木の本数
    random_state=42    # 再現性のためのシード
)
model.fit(X_train_scaled, y_train)
print('\n学習完了！')

---
## 4.4 評価：モデルの性能を測る

### 📢 講師メモ
評価は「単に正解率を出す」だけでなく「どう間違えたか」を見ることが重要です。
特に医療・金融などリスクが非対称な業務では、混同行列の読み方が命取りになります。

In [ ]:
# 予測と正解率
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f'正解率（Accuracy）: {acc:.1%}')

In [ ]:
# 混同行列（Confusion Matrix）
cm = confusion_matrix(y_test, y_pred)
class_names = ['Barolo(0)', 'Grignolino(1)', 'Barbera(2)']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm,
            annot=True, fmt='d',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            linewidths=0.5)
ax.set_title('混同行列（Confusion Matrix）', fontsize=14)
ax.set_xlabel('予測ラベル', fontsize=12)
ax.set_ylabel('正解ラベル', fontsize=12)
plt.tight_layout()
plt.show()

print('=== 読み方 ===')
print('対角成分（左上〜右下）: 正しく予測できた件数')
print('対角以外: 誤分類した件数')
print()
print('例：行[0]列[1] = Barolo を Grignolino と誤分類した件数')

### 📢 混同行列の読み方スクリプト（受講者向けに口頭で説明）

```
「行が正解ラベル、列が予測ラベルです。

対角線（左上から右下）の数字が大きいほど正解が多い。

例えば：
- 行0・列1の数字 = 本当は Barolo なのに Grignolino と予測した件数
- 行1・列0の数字 = 本当は Grignolino なのに Barolo と予測した件数

どの品種が最も間違われやすいか？を見ることで、
モデルのどこを改善すべきか判断できます。」
```

In [ ]:
# 詳細な評価レポート
print('=== 評価レポート（classification_report）===')
print(classification_report(y_test, y_pred, target_names=class_names))

### 📢 Precision・Recall・F1 の説明

```
Precision（適合率）:
  「Baroloだと予測した件数のうち、本当に Barolo だった割合」
  高い → 誤検知（FP）が少ない
  重視する場面：迷惑メールフィルタ（正常メールを迷惑と誤判定したくない）

Recall（再現率）:
  「本当に Barolo だった件数のうち、Baroloと検出できた割合」
  高い → 見逃し（FN）が少ない
  重視する場面：がん検診（陽性の見逃しが命取り）

F1-score:
  Precision と Recall の調和平均（バランスの指標）
```

---
## 4.5 特徴量重要度：何が予測に貢献しているか

### 📢 講師メモ
Ch.2 の EDA で立てた仮説との答え合わせをします。
受講者の仮説と照合してください。
> 「Ch.2 でこのような仮説を立てた方はいますか？今から答え合わせです。」

In [ ]:
# 特徴量重要度の取得と可視化
importances = pd.Series(model.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
colors_list = ['#e74c3c' if imp > 0.1 else 'steelblue' for imp in importances.values]
importances.plot(kind='barh', ax=ax, color=colors_list)
ax.axvline(x=1/13, color='gray', linestyle='--', alpha=0.7,
           label=f'均等な場合（{1/13:.3f}）')
ax.set_title('特徴量重要度（Feature Importances）', fontsize=14)
ax.set_xlabel('重要度')
ax.legend()
plt.tight_layout()
plt.show()

print('重要度トップ3：')
print(importances.sort_values(ascending=False).head(3).round(3))

### 📢 特徴量重要度の解説スクリプト

```
「proline が最も重要度が高い → Ch.2 の箱ひげ図で Barolo だけ圧倒的に高かった
 alcohol も上位 → ヒストグラムで Barolo が右寄りだった
 color_intensity も重要 → 色の濃さでも品種が分かれていた

EDA の仮説が、モデルでも裏付けられました。
これが「EDA → モデル → 解釈」の流れの意味です。」

点線（均等な場合）= 1/13 ≈ 0.077
これより高い変数は「平均以上に重要」と判断できる
```

---
## 🔑 演習の解答・解説

### 📢 演習進行のポイント
- 問1はコードより「なぜ精度が変わるか」の考察が重要
- 問2は「スケーリングの効果」を数値で体感させる
- 問3の記述問題は必ず発表させる
- 問4（発展）は特徴量選択（Feature Selection）の概念の入口

### 問1：ハイパーパラメータを変えて精度の変化を確認する

In [ ]:
# ✅ 問1：解答
n_estimators_list = [10, 50, 100, 200]
accuracies = []

for n in n_estimators_list:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    acc_n = accuracy_score(y_test, pred)
    accuracies.append(acc_n)
    print(f'n_estimators={n:3d}: Accuracy = {acc_n:.1%}')

# 折れ線グラフ
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_estimators_list, [a * 100 for a in accuracies],
        marker='o', linewidth=2, markersize=8, color='steelblue')
for n, a in zip(n_estimators_list, accuracies):
    ax.annotate(f'{a:.1%}', (n, a * 100), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=11)
ax.set_title('n_estimators と正解率の関係')
ax.set_xlabel('n_estimators（決定木の本数）')
ax.set_ylabel('正解率（%）')
ax.set_ylim(80, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**📢 解説：**
- 本数が少ない（10本）と精度が不安定
- 100本以上になると精度が頭打ちになりやすい
- これを**収穫逓減**という：コストを増やしても効果が増えなくなるポイント
- 実務では 100〜500 本が多く使われる

> 「このようにパラメータを変えて精度を比較することを『ハイパーパラメータチューニング』といいます。
> 発展的には Optuna などのライブラリで自動化できますが、今日はこの感覚を掴むことが目的です。」

### 問2：スケーリングなしの場合と比較する

In [ ]:
# ✅ 問2：解答

# スケーリングなしでモデルを学習
model_noscale = RandomForestClassifier(n_estimators=100, random_state=42)
model_noscale.fit(X_train, y_train)  # スケーリングなしの生データ
y_pred_noscale = model_noscale.predict(X_test)
acc_noscale = accuracy_score(y_test, y_pred_noscale)

print('=== スケーリングあり vs なしの比較 ===')
print(f'スケーリングあり: Accuracy = {acc:.1%}')
print(f'スケーリングなし: Accuracy = {acc_noscale:.1%}')
print()

# 特徴量重要度も比較
imp_scaled   = pd.Series(model.feature_importances_,         index=wine.feature_names)
imp_noscaled = pd.Series(model_noscale.feature_importances_, index=wine.feature_names)

comparison = pd.DataFrame({
    'スケーリングあり': imp_scaled,
    'スケーリングなし': imp_noscaled
}).sort_values('スケーリングあり', ascending=False).round(3)
print('特徴量重要度の比較（上位5）:')
print(comparison.head())

**📢 解説：**
- ランダムフォレストの正解率はスケーリングの有無で大きくは変わらない
- なぜなら、決定木は「閾値との比較」で分岐するため、値のスケールが影響しにくい
- **ただし**、距離ベースのアルゴリズム（KNN、SVM、ロジスティック回帰など）はスケーリングが必須

> 「今日はランダムフォレストを使っているので影響が小さいです。
> しかし「スケーリングは必ずする」を習慣にしておくと、アルゴリズムを変えたときに安全です。」

### 問3：混同行列を読み解く（記述問題）

In [ ]:
# ✅ 問3：混同行列の数値確認
print('=== 混同行列の数値 ===')
print('行 = 正解ラベル / 列 = 予測ラベル')
print()
cm_df = pd.DataFrame(cm,
                     index=[f'実際: {c}' for c in class_names],
                     columns=[f'予測: {c}' for c in class_names])
print(cm_df)
print()

# 各品種の Recall を計算
for i, class_name in enumerate(class_names):
    recall = cm[i, i] / cm[i, :].sum()
    print(f'{class_name} の Recall（検出率）: {recall:.1%}')

**📢 模範解答：**

**Q1. 最も誤分類されやすい品種は？**
> 混同行列を見て、対角以外の数字が最も多い行の品種。
> 結果によって異なるが、一般に Grignolino と Barbera は特徴が似ているため混同されやすい。

**Q2. Barolo が Grignolino に誤分類される場合の実務的問題（例）：**
> 「Barolo は高級ワインであることが多い。Grignolino に誤分類されると価格が低く設定され、損失が出る可能性がある。
> 偽物検査の場合は品質問題に繋がる。」

**Q3. 正解率が高くても使えないケース：**
> 「不均衡データ（例：100件中95件が陰性）では、全部陰性と予測するだけで正解率95%になる。
> しかし陽性を全て見逃している（Recall = 0%）ため、モデルとして機能しない。」

### 問4（発展）：重要度が高い上位3特徴量だけでモデルを作り直す

In [ ]:
# ✅ 問4：解答

# 重要度上位3特徴量を選択
top3_features = importances.sort_values(ascending=False).head(3).index.tolist()
print('重要度トップ3の特徴量:', top3_features)

# 上位3特徴量だけでデータを準備
X_train_top3 = X_train[top3_features]
X_test_top3  = X_test[top3_features]

# スケーリング
scaler_top3 = StandardScaler()
X_train_top3_scaled = scaler_top3.fit_transform(X_train_top3)
X_test_top3_scaled  = scaler_top3.transform(X_test_top3)

# モデル学習
model_top3 = RandomForestClassifier(n_estimators=100, random_state=42)
model_top3.fit(X_train_top3_scaled, y_train)

# 評価
y_pred_top3 = model_top3.predict(X_test_top3_scaled)
acc_top3 = accuracy_score(y_test, y_pred_top3)

print()
print(f'全特徴量（13列）: Accuracy = {acc:.1%}')
print(f'上位3特徴量のみ:  Accuracy = {acc_top3:.1%}')

**📢 解説：**
- 上位3特徴量だけで13特徴量とほぼ同等の精度が出ることが多い
- これが「**特徴量選択（Feature Selection）**」の考え方
- 利点：モデルが単純になる・計算が速くなる・解釈しやすい・過学習リスクが下がる

> 「実務では「精度を下げずに変数を減らせるか」を検討することが重要です。
> 変数が少ないほどデータ収集コストが下がり、運用が楽になります。」

---
## ✅ チャプターのまとめ（講師用コメント付き）

| ステップ | コード | 最重要ポイント |
|---------|--------|---------------|
| データ分割 | `train_test_split(..., stratify=y)` | stratify を忘れない |
| スケーリング（学習用） | `scaler.fit_transform(X_train)` | fit は訓練データのみ！ |
| スケーリング（評価用） | `scaler.transform(X_test)` | fit_transform は絶対NG |
| モデル学習 | `model.fit(X_train_scaled, y_train)` | — |
| 予測 | `model.predict(X_test_scaled)` | — |
| 評価 | `accuracy_score + confusion_matrix` | 必ずセットで確認 |
| 特徴量重要度 | `model.feature_importances_` | EDAの仮説と照合する |

### 📢 Ch.5 への橋渡し
> 「Ch.1〜4 で学んだ全ての内容を使って、新しいデータで自力でやってみます。
> 今度は医療データ（乳がん診断）です。「正解率が高ければOK」ではない、
> 見逃しのリスクを考えた評価を特に意識してください。」